In [18]:
!pip install -q plotly pandas

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
from IPython.display import display

print("Libraries imported successfully!")


Libraries imported successfully!


In [19]:
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


Saving netflix_titles.csv to netflix_titles (1).csv
Dataset loaded successfully!
Rows and columns: (8807, 12)

Column names:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [20]:
df["date_added"] = pd.to_datetime(
    df["date_added"].str.strip(),
    format="%B %d, %Y",
    errors="coerce"
)

for col in ["rating", "country", "listed_in", "title"]:
    df[col] = df[col].fillna("Unknown")

df["duration_value"] = pd.to_numeric(
    df["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

df["genre_count"] = df["listed_in"].apply(
    lambda x: 0 if x == "Unknown" else len(x.split(","))
)

df["cast_count"] = df["cast"].fillna("").apply(
    lambda x: len([name for name in x.split(",") if name.strip()])
)

df["director_count"] = df["director"].fillna("").apply(
    lambda x: len([name for name in x.split(",") if name.strip()])
)

df["release_year"] = pd.to_numeric(
    df["release_year"], errors="coerce"
)

print("Data preparation completed!")
print("Missing values:")
print(df.isnull().sum())
display(df.head())


Data preparation completed!
Missing values:
show_id              0
type                 0
title                0
director          2634
cast               825
country              0
date_added          10
release_year         0
rating               0
duration             3
listed_in            0
description          0
duration_value       3
genre_count          0
cast_count           0
director_count       0
dtype: int64


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_value,genre_count,cast_count,director_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,1,0,1
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2.0,3,19,0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,1.0,3,9,1
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",1.0,2,0,0
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2.0,3,8,0


In [21]:
movies = df[
    (df["type"] == "Movie") &
    (df["duration_value"].notna()) &
    (df["release_year"].notna())
].copy()

animation_df = movies[
    movies["release_year"].between(1995, 2021)
].copy()

movies = movies.sort_values("release_year")

print("Total Netflix titles:", len(df))
print("Movies with duration:", len(movies))
print("Movies from 1995 to 2021:", len(animation_df))
print("Preparation for charts completed!")


Total Netflix titles: 8807
Movies with duration: 6128
Movies from 1995 to 2021: 5794
Preparation for charts completed!


In [22]:
fig1 = px.histogram(
    movies,
    x="release_year",
    nbins=40,
    title="1. Distribution of Netflix Movie Release Years",
    labels={"release_year": "Release Year"},
    color_discrete_sequence=["royalblue"],
    hover_data=["title"]
)

fig1.update_layout(
    xaxis_rangeslider_visible=True,
    xaxis_title="Release Year",
    yaxis_title="Number of Movies",
    template="plotly_white"
)

fig1.show()


In [23]:
fig2 = px.scatter(
    movies,
    x="release_year",
    y="duration_value",
    color="rating",
    hover_data=["title", "country", "date_added"],
    title="2. Movie Release Year vs Duration by Rating",
    labels={
        "release_year": "Release Year",
        "duration_value": "Duration (minutes)",
        "rating": "Content Rating"
    },
    opacity=0.7
)

fig2.update_layout(template="plotly_white")
fig2.show()


In [24]:
fig2_log = px.scatter(
    movies[movies["duration_value"] > 0],
    x="release_year",
    y="duration_value",
    color="rating",
    hover_data=["title", "country", "date_added"],
    log_y=True,
    title="2B. Movie Duration vs Release Year (Log Scale)"
)

fig2_log.update_layout(template="plotly_white")
fig2_log.show()


In [25]:
fig3 = px.box(
    movies,
    x="rating",
    y="duration_value",
    color="rating",
    points="all",
    hover_data=["title", "release_year"],
    title="3. Movie Duration Distribution by Content Rating",
    labels={
        "rating": "Content Rating",
        "duration_value": "Duration (minutes)"
    }
)

fig3.update_layout(
    template="plotly_white",
    showlegend=False
)

fig3.show()


In [26]:
fig4 = px.scatter_3d(
    movies,
    x="release_year",
    y="duration_value",
    z="genre_count",
    color="rating",
    hover_data=["title", "country"],
    title="4. 3D Analysis of Netflix Movies",
    labels={
        "release_year": "Release Year",
        "duration_value": "Duration (minutes)",
        "genre_count": "Number of Genres"
    }
)

fig4.update_layout(template="plotly_white")
fig4.show()


In [27]:
fig5 = px.scatter(
    animation_df,
    x="duration_value",
    y="genre_count",
    animation_frame="release_year",
    size="cast_count",
    color="rating",
    hover_name="title",
    hover_data=["country", "listed_in"],
    size_max=25,
    range_x=[
        0,
        animation_df["duration_value"].max() + 10
    ],
    range_y=[
        0,
        animation_df["genre_count"].max() + 1
    ],
    title="5. Animated Netflix Movie Analysis by Year",
    labels={
        "duration_value": "Duration (minutes)",
        "genre_count": "Number of Genres",
        "release_year": "Release Year",
        "cast_count": "Number of Cast Members"
    }
)

fig5.update_layout(template="plotly_white")
fig5.show()



In [28]:
dropdown_df = movies.copy()

metrics = [
    ("duration_value", "Duration (minutes)"),
    ("genre_count", "Number of Genres"),
    ("cast_count", "Number of Cast Members")
]

fig6 = go.Figure()

for column, label in metrics:
    fig6.add_trace(
        go.Scatter(
            x=dropdown_df["release_year"],
            y=dropdown_df[column],
            mode="markers",
            name=label,
            text=dropdown_df["title"],
            customdata=dropdown_df[["country", "rating"]].values,
            hovertemplate=(
                "Title: %{text}<br>"
                "Release Year: %{x}<br>"
                + label + ": %{y}<br>"
                + "Country: %{customdata[0]}<br>"
                + "Rating: %{customdata[1]}"
                + "<extra></extra>"
            ),
            visible=(column == "duration_value")
        )
    )

buttons = []

for i, (column, label) in enumerate(metrics):
    visibility = [j == i for j in range(len(metrics))]

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "yaxis": {"title": label},
                    "title": "6. Release Year vs " + label
                }
            ]
        )
    )

fig6.update_layout(
    title="6. Release Year vs Duration (minutes)",
    xaxis_title="Release Year",
    yaxis_title="Duration (minutes)",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.02,
            y=1,
            showactive=True
        )
    ],
    template="plotly_white"
)

fig6.show()
